# Reproduce the paper's tables and figures from the shipped data

Every table and figure in *How Relations Impose Embedding-Space Structure* is regenerated here from `data/`,
on a CPU, with no API key or GPU. Each cell runs one script from `analysis/` (output into `outputs/`) and then
shows the numbers or figure beside its paper reference. The scripts can equally be run from a shell.

**Data.** The git repository holds the core of `data/` (vectors, concept indices, judge results, training text).
The rest (steered generations, per-row judge scores, the 32B models' vectors, SAE activations; about 0.9 GB in
all) is in the Hugging Face dataset
[octopus123/embedded_relationships](https://huggingface.co/datasets/octopus123/embedded_relationships) and is
downloaded automatically the first time a script needs it (`relgeom/data.py`; or run `python -m relgeom.data`).

**Scope.** `SCOPE = "main"` regenerates the main-text results (Table 2, Figures 3 and 4; Tables 1, 3 and 4
are fixed inputs, Figures 1 and 2 illustrations). `SCOPE = "all"` adds every appendix table and figure.
The permutation nulls (5,000 draws each) dominate the run time: about 30 minutes for `"main"` and
2 hours 15 minutes for `"all"` on an 8-core CPU node (the SAE analysis alone takes about 40 minutes).

In [ ]:
SCOPE = "main"          # "main" (main text) or "all" (main text + appendix)

import os, subprocess, sys, time
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (ROOT / "relgeom").is_dir(), f"run this notebook from the repository (cwd={Path.cwd()})"
TABLES, FIGURES = ROOT / "outputs/tables", ROOT / "outputs/figures"
pd.set_option("display.width", 200, "display.max_columns", 30)


def run(script, model=None, tail=8):
    """Run analysis/<script> (optionally for another model: RELGEOM_MODEL) and print the end of its output."""
    env = dict(os.environ, MPLBACKEND="Agg")
    env.pop("RELGEOM_MODEL", None)
    if model:
        env["RELGEOM_MODEL"] = model
    t = time.time()
    r = subprocess.run([sys.executable, str(ROOT / "analysis" / script)], cwd=ROOT, env=env,
                       capture_output=True, text=True)
    lines = (r.stdout + r.stderr).strip().splitlines()
    print(f"[{script}{' ' + model if model else ''}] {time.time() - t:.0f}s")
    print("\n".join(lines[-tail:]) if tail else "")
    if r.returncode:
        raise RuntimeError(f"{script} failed:\n" + "\n".join(lines[-40:]))


def table(name, model=None):
    return pd.read_csv(TABLES / (f"{model}/" if model else "") / name)


def figure(name, width=700):
    display(Image(filename=str(FIGURES / name), width=width))

MODELS = ["olmo31_32b", "qwen3_32b"]
APPENDIX = SCOPE == "all"
print("scope:", SCOPE, "| outputs ->", ROOT / "outputs")

## 1. Temperature: water words vs numeric temperatures (Section 4)

**Table 2** (layer 10) and **Tables 6-7** (all layers): the move RSA between the 49 water-word moves and their
numeric counterparts, against a word-permutation null, and the direction statistics. The ladders and word
temperatures (**Table 1**) are in `relgeom/ladders.py` and `relgeom/harness.py`.

In [ ]:
run("temperature/moves_rsa.py", tail=4)
mv = table("move_direction_stats_by_layer.csv")
cols = ["layer", "unit", "rho", "null_mean", "null_p95", "p_word", "cross_cos_mean", "within_water_cos", "within_temp_cos"]
print("Table 2 (layer 10):"); display(mv[mv.layer == 10][cols])
if APPENDIX:
    print("Tables 6-7 (all layers):"); display(mv[cols])

**Table 12** (appendix): the positions RSA between the two domains' position matrices, all layers.

In [ ]:
if APPENDIX:
    run("temperature/positions_rsa.py", tail=0)
    display(table("node_rsa_by_layer_unit.csv")[["layer", "unit", "spearman", "z", "perm_p"]])

## 2. Analogies (Section 5)

**Figure 3**: move agreement for IMPORTANCE IS SIZE, ARGUMENT IS WAR, IDEAS ARE FOOD and the professional frames,
at layers 5, 10, 15, 20, against the correspondence-permutation null. The mappings (**Tables 3-4**) are in
`relgeom/analogies.py`.

In [ ]:
run("analogies/analogy_rsa.py", tail=0)
r = table("analogy_rsa.csv")
moves = r[r.kind == "moves"]
fig3 = (moves.groupby(["experiment", "layer"])
        .agg(rho=("rho", "mean"), null_p95=("null_p95", "mean"), p=("p", "median")).round(3))
display(fig3.unstack("layer"))

**Table 8** (direction statistics for the analogies) and **Table 9** (the ten profession pairs at layer 10),
appendix.

In [ ]:
if APPENDIX:
    run("analogies/direction_stats.py", tail=0)
    display(table("analogy_direction_stats.csv").head(8))
    run("analogies/frames_pairs.py", tail=20)

## 3. Other models (Section 6)

Each analysis is re-run on OLMo-3.1-32B and Qwen3-32B vectors (`RELGEOM_MODEL=<tag>`), at the same relative depth,
then collected by `models/cross_model_rsa.py` into **Figure 4** and **Tables 10 and 13**.

In [ ]:
for m in MODELS:
    run("temperature/moves_rsa.py", model=m, tail=0)
    run("analogies/analogy_rsa.py", model=m, tail=0)
    if APPENDIX:
        run("temperature/positions_rsa.py", model=m, tail=0)
run("models/cross_model_rsa.py", tail=0)
s = table("cross_model_summary.csv")
s["sig"] = s.p < .05


def counts(kind):
    # layers clearing p < .05 per experiment and model; frames: range of #profession pairs (of 10) clearing
    d = s[s.analysis.str.startswith(kind)]
    main = d[d.experiment != "frames_v1"]
    out = (main.groupby(["experiment", "analysis", "model"]).sig
           .agg(lambda v: f"{int(v.sum())}/{len(v)}").unstack("model"))
    fr = d[d.experiment == "frames_v1"].groupby("model").n_sig.agg(lambda v: f"{int(v.min())}-{int(v.max())}")
    out.loc[("frames_v1", kind), :] = fr
    return out

print("Table 10: depth-matched layers at which the move agreement clears its null")
display(counts("moves"))
if APPENDIX:
    print("Table 13: the same for positions")
    display(counts("positions"))

## 4. Steering efficacy (Appendix A)

**Figure 5**: judge scores for the temperature vectors. **Figure 6**: the judge-free check for the target
temperature. **Figure 7**: judge scores for the analogy vectors. **Table 5**: steering efficacy in all three
models on the shared grid (9 factors x first 6 held-out prompts).

In [ ]:
if APPENDIX:
    run("steering/temperature_judge_scores.py", tail=3)
    run("steering/temperature_parse_check.py", tail=3)
    run("steering/analogy_judge_scores.py", tail=6)
    run("models/cross_model_steering.py", tail=14)

## 5. Sensitivity: removing the least effective vectors (Appendix G)

Each vector gets one judge score (Overall, max over factors). The temperature and analogy analyses are re-run
after dropping the fraction f of lowest-scoring items, for every model: **Figures 16-17**, **Table 11**.

In [ ]:
if APPENDIX:
    for m in [None] + MODELS:
        run("sensitivity/build_scores.py", model=m, tail=0)
        run("sensitivity/sweep_temperature.py", model=m, tail=0)
        run("sensitivity/sweep_analogies.py", model=m, tail=2)
    run("sensitivity/cross_model_table.py", tail=12)

## 6. SAE features (Appendix H)

**Figure 18**: top-20 GemmaScope features shared by each water word and its matched Celsius temperature.

In [ ]:
if APPENDIX:
    run("sae/activation_view.py", tail=0)
    o = table("act_overlap.csv")
    display(o[(o.layer == 10) & (o.k == 20)][["family", "jacc_matched", "jacc_null", "auc"]])

## 7. Figures

In [ ]:
run("figures/main_rsa_figures.py", tail=0)
print("Figure 3"); figure("analogy_rsa_moves.png", 420)
print("Figure 4"); figure("models_verification_moves.png", 900)
if APPENDIX:
    for s_ in ["analogy_steering_figure.py", "appendix_figures.py", "sae_figure.py"]:
        run("figures/" + s_, tail=0)
    for n, f in [("Figure 5", "temp_steering_eval.png"), ("Figure 6", "temp_steering_parse.png"),
                 ("Figure 7", "analogy_steering.png"), ("Figure 15", "temp_rsa_by_layer_moves.png"),
                 ("Figure 16", "temp_sweep_moves.png"), ("Figure 17", "analogy_sweep_moves.png"),
                 ("Figure 18", "sae_overlap_L10.png")]:
        print(n); figure(f)